In [23]:
import gradio as gr
import sqlite3
import os 
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [24]:
gemini = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=os.getenv("gemini_url"))
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"
system = """
You are an system assistant for a restaurant called Ember Resort.
You should provide short and precise answers, in 3 sentences.
If you don't have the answer, don't say anything.
""" 

# meal_prices = {
#     "prawns tempura": "$50", 
#     "naan and butter chicken": "$9.27",
#     "fries and chicken sandwich": "$7.34",
#     "blueberry milkshake": "$20",
#     "pancakes": "$15", 
#     "latte": "$7.50"
# }

In [25]:
DB = "data/ember_restaurant.db"  

with sqlite3.connect(DB) as conn: 
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS MEALS(meal TEXT PRIMARY KEY, price REAL)")
    conn.commit()

In [26]:
def get_meal_prices(meal): 
    with sqlite3.connect(DB) as conn: 
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM MEALS WHERE meal= ?", (meal.lower(),))
        result = cursor.fetchone()

    return f"The price of {meal} is {result[0]}" if result else None

def get_meal_names(): 
    result = []
    with sqlite3.connect(DB) as conn: 
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM MEALS")
        meals = cursor.fetchall()

    for meal in meals: 
        result.append(meal)

    return result

In [27]:
name_function = {
    "name": "get_meal_names", 
    "description": "gets the names of meals and their prices that are available in list form", 
    "parameters": {
        "type": "object", 
        "properties": {
            "meals": {
                "type": "string", 
                "description": "names of meals and prices"
            }
        },
        "required": ["meals"], 
        "additionalParameters": False
    }
}

In [28]:
price_function = {
    "name": "get_meal_prices", 
    "description": "gets the prices of meals ordered that are available",
    "parameters": {
        "type": "object", 
        "properties": {
            "meal": {
                "type": "string", 
                "description": "the price of a meal a customer has ordered"
            }
        },
        "required": ["meal"],
        "additionalParameters": False
    }
}

In [29]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": name_function}
]

In [30]:
def handleToolCall(message): 
    response = []
    for tool_call in message.tool_calls: 
        if(tool_call.function.name == "get_meal_prices"): 
            arguments = json.loads(tool_call.function.arguments)
            meal = arguments.get('meal') 
            price_details = get_meal_prices(meal)
            response.append({
                "role": "tool", 
                "content": price_details, 
                "tool_call_id": tool_call.id
            })
        if(tool_call.function.name == "get_meal_names"): 
            meals = get_meal_names()
            for meal in meals: 
                response.append({
                    "role": "tool", 
                    "content": meal[0], 
                    "tool_call_id": tool_call.id
                })

    return response

In [31]:
def prompt_in_chatbot(prompt, history): 
    msg = []
    msg.append({"role": "user", "content": prompt})
    return "", history + msg

In [32]:
def ai_chat(history): 
    msg = []
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    msg.append(history)
    msg.append({"role": "system", "content": system})
    response = gemini.chat.completions.create(model=MODEL, messages=msg, tools=tools)

    while response.choices[0].finish_reason == "tool_calls": 
        message = response.choices[0].message
        rsp = handleToolCall(message)
        msg.append(message)
        msg.extend(rsp)
        response = gemini.chat.completions.create(model=MODEL, messages=msg, tools=tools)

    reply = response.choices[0].message.content

    history.append({"role": "assistant", "content": reply})

    return history

In [ ]:
with gr.Blocks(theme=gr.Theme.from_hub("hmb/wii")) as ui: 
    gr.Markdown("<h1>Ember Restaurant Ai assistant</h1>".upper(), elem_classes="title")
    chatbot = gr.Chatbot(label="Model: Gemini")
    chat = gr.Textbox(
        placeholder="Chat with AI Assistant", 
        container=False, 
        elem_classes="chat_message", 
        submit_btn=True, 
        max_lines=5, 
        lines=1,
    )

    chat.submit(fn=prompt_in_chatbot, inputs=[chat, chatbot], outputs=[chat, chatbot]).then(
        fn=ai_chat,
        inputs=chatbot,
        outputs=[chatbot]
    )

ui.launch(css_paths="styles.css")

c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\gradio\themes\base.py:163: UserWarning: This theme was created for Gradio 6.11.0, but you are using Gradio 6.25.0. Some styles may not work as expected.
  warnings.warn(
C:\Users\danie\AppData\Local\Temp\ipykernel_26252\671600066.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.Theme.from_hub("hmb/wii")) as ui:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\gradio\routes.py", line 692, in main
  File "c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\starlette\templating.py", line 148, in TemplateResponse
    template = self.get_template(name)
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\starlette\templating.py", line 115, in get_template
    return self.env.get_template(name)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\jinja2\environment.py", line 1016, in get_template
  File "c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\jinja2\environment.py", line 975, in _load_template
  File "c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\jinja2\loaders.py", line 126, in load
  File "c:\Users\danie\anaconda3\envs\.Llm_env\Lib\site-packages\jinja2\loaders.py", line 209